# Fantasy Football: Model Comparison and Evaluation
### CSCI 5502 | Milestone 3

## 1. Objective

The goal of this notebook is to compare the machine learning models developed by our team and evaluate how accurately they predict NFL fantasy football points.

Our main focus is to compare two existing modeling approaches:

1. **Global XGBoost:** A single XGBoost model trained on players across all positions.
2. **Clustered XGBoost:** Three separate XGBoost models trained on position groups identified through clustering:
   - Running Backs (RB)
   - Quarterbacks and Tight Ends (QB + TE)
   - Wide Receivers and Kickers (WR + K)

We will also compare both approaches against ESPN's fantasy point projections as an external benchmark.

## 2. Evaluation Metrics

We will use three regression metrics:

- **Mean Absolute Error (MAE):** Average absolute difference between predicted and actual fantasy points.
- **Root Mean Squared Error (RMSE):** Measures prediction error while penalizing larger mistakes more heavily.
- **R-squared (R²):** Measures how well the predictions explain variation in actual fantasy scores.

## 3. Comparison Strategy

To ensure a fair comparison, we will:

1. Use the same player-game observations for both models.
2. Use chronological training, validation, and testing splits.
3. Check for target leakage and features unavailable before the game.
4. Compare overall prediction performance.
5. Compare model performance across QB, RB, WR, TE, and K.
6. Analyze which model performs better than ESPN and where each model struggles.

The objective is not simply to achieve the lowest error, but to determine which modeling approach is more reliable for predicting future fantasy football performance.


In [1]:

# Data manipulation
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# XGBoost
import xgboost as xgb

sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")
print("XGBoost version:", xgb.__version__)


All libraries imported successfully!
XGBoost version: 3.4.1


In [3]:

# Load datasets from the data folder
data_dir = Path("data")

df_2024 = pd.read_csv(data_dir / "df_2024_final.csv")
df_2025 = pd.read_csv(data_dir / "df_2025_final.csv")
df_2026 = pd.read_csv(data_dir / "df_2026_final.csv")

# Assign season labels
df_2024["season"] = 2024
df_2025["season"] = 2025
df_2026["season"] = 2026

# Combine the datasets
combined = pd.concat(
    [df_2024, df_2025, df_2026],
    ignore_index=True
)

# Check dataset sizes
for year, df in [(2024, df_2024),
                 (2025, df_2025),
                 (2026, df_2026)]:
    print(f"{year}: {df.shape[0]} rows, {df.shape[1]} columns")

print("\nCombined shape:", combined.shape)


2024: 7522 rows, 84 columns
2025: 7572 rows, 84 columns
2026: 4440 rows, 75 columns

Combined shape: (19534, 86)


In [4]:

# Check which columns differ between seasons
cols_2024 = set(df_2024.columns)
cols_2025 = set(df_2025.columns)
cols_2026 = set(df_2026.columns)

print("Columns in 2024 but not 2026:")
print(sorted(cols_2024 - cols_2026))

print("\nColumns in 2026 but not 2024:")
print(sorted(cols_2026 - cols_2024))

# Check important columns for our prediction problem
important_cols = [
    "player_id",
    "player",
    "position",
    "season",
    "week",
    "fantasy_points",
    "projected_points"
]

print("\nImportant columns available:")
for col in important_cols:
    print(f"{col}: {col in combined.columns}")

# Check missing values in important columns
print("\nMissing values by season:")
print(
    combined.groupby("season")[
        important_cols
    ].apply(lambda x: x.isna().sum())
)

# Check sample records
display(combined[important_cols].head())


Columns in 2024 but not 2026:
['assistTackles', 'avgInterceptionYards', 'fumblesRecovered', 'interceptionTouchdowns', 'interceptionYards', 'longInterception', 'passesDefended', 'soloTackles', 'stuffYards', 'stuffs', 'totalTackles']

Columns in 2026 but not 2024:
['pro_team_id', 'season_type']

Important columns available:
player_id: True
player: True
position: True
season: True
week: True
fantasy_points: True
projected_points: True

Missing values by season:
        player_id  player  position  season  week  fantasy_points  \
season                                                              
2024            0       0         0       0     0               0   
2025            0       0         0       0     0               0   
2026            0       0         0       0     0             554   

        projected_points  
season                    
2024                   3  
2025                  30  
2026                 183  


,player_id,player,position,season,week,fantasy_points,projected_points
0,4374302,Amon-Ra St. Brown,WR,2024,18,13.7,18.263253
1,4374302,Amon-Ra St. Brown,WR,2024,17,20.0,17.459376
2,4374302,Amon-Ra St. Brown,WR,2024,16,19.0,17.948622
3,4374302,Amon-Ra St. Brown,WR,2024,15,37.3,17.177120
4,4374302,Amon-Ra St. Brown,WR,2024,14,10.3,17.494361



### Interpretation of Data Inspection

After inspecting the datasets, I found some differences in the columns across seasons. The 2024 dataset contains several defensive statistics that are missing from 2026, while the 2026 dataset includes two additional columns (`pro_team_id` and `season_type`). Since our goal is to compare models fairly, we will need to make sure the models use consistent and relevant features.

All important columns, including player ID, position, season, week, fantasy points, and ESPN projections, are available across the datasets.

I also found that 2024 and 2025 have no missing actual fantasy scores, but 2026 has 554 missing values. ESPN projections are missing for 3 records in 2024, 30 in 2025, and 183 in 2026.

Before evaluating the models, we need to investigate the missing 2026 scores and exclude observations without known outcomes from the evaluation. This will help ensure that our model comparisons are accurate and consistent.



## 5. Investigating Missing Fantasy Scores

The 2026 dataset contains 554 missing fantasy scores. Before preparing the test dataset, we need to investigate which weeks contain these missing values.

We will also check ESPN projection availability and look for duplicate player-week records.

This will help us identify which observations are suitable for evaluating our models.


In [5]:

# Summarize available scores by week
weekly_summary = (
    df_2026.groupby("week")
    .agg(
        total_players=("player_id", "count"),
        actual_scores=("fantasy_points", "count"),
        espn_projections=("projected_points", "count")
    )
)

# Calculate missing scores and projections
weekly_summary["missing_actual"] = (
    weekly_summary["total_players"]
    - weekly_summary["actual_scores"]
)

weekly_summary["missing_espn"] = (
    weekly_summary["total_players"]
    - weekly_summary["espn_projections"]
)

display(weekly_summary)

# Check which weeks contain missing actual scores
print("\nMissing actual scores by week:")
print(
    df_2026[df_2026["fantasy_points"].isna()]
    ["week"].value_counts().sort_index()
)

# Check duplicate player-week observations
duplicates = df_2026.duplicated(
    subset=["player_id", "season", "week"]
).sum()

print("\nDuplicate player-week records:", duplicates)


,total_players,actual_scores,espn_projections,missing_actual,missing_espn
week,,,,,
1,448,448,438,0,10
2,467,467,454,0,13
3,470,470,455,0,15
4,463,463,443,0,20
5,424,424,411,0,13
6,395,395,382,0,13
7,393,393,378,0,15
8,401,401,385,0,16
9,425,425,402,0,23



Missing actual scores by week:
week
10    554
Name: count, dtype: int64

Duplicate player-week records: 0



### Block 5: Interpretation

After examining the 2026 dataset, I found that all 554 missing fantasy scores belong to Week 10. Weeks 1 through 9 have complete actual fantasy scores, which gives us 3,886 player-game observations that can potentially be used for evaluation.

Week 10 contains ESPN projections for 509 players, but none of the actual scores are available. This suggests that the data was collected before those games were completed.

I also checked for duplicate player-season-week records and found none.

Based on these findings, we will use completed games from Weeks 1 through 9 of 2026 as our test period and exclude Week 10 from performance evaluation.

For comparisons with ESPN, we will only use observations where both the actual fantasy score and ESPN projection are available. This ensures a fair comparison across models.

One important detail is that our models predict the next game using historical information. We will therefore need to align each prediction with the correct target game before calculating evaluation metrics.


In [ ]:
rb2e dfwdgl˚fg dsxaz;